## Intialization

In [ ]:
import pandas as pd
import numpy as np
import spacy
import shap
import re
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

import joblib
import json
import urllib.request
import zipfile
import os
import pickle

from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

c:\Users\julia\SY2627\Projects\country_classification_on_wine_reviews\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "cpu"
)

print(f"Using {device} device")

Using cpu device


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zynicide/wine-reviews")

print("Path to dataset files:", path)

100%|██████████| 50.9M/50.9M [00:02<00:00, 18.0MB/s]

Extracting files...


Path to dataset files: C:\Users\julia\.cache\kagglehub\datasets\zynicide\wine-reviews\versions\4


In [4]:
df = pd.read_csv(path + "/winemag-data-130k-v2.csv")
df.head()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            92506 non-null   str    
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  str    
 7   region_1               108724 non-null  str    
 8   region_2               50511 non-null   str    
 9   taster_name            103727 non-null  str    
 10  taster_twitter_handle  98758 non-null   str    
 11  title                  129971 non-null  str    
 12  variety                129970 non-null  str    
 13  winery                 129971 non-null  str    
dtypes: float64(1), int64(2), str(11)
memory usage: 

In [7]:
print(f"Shape: {df.shape}")
print(f"Number of Missing Values: {df.isnull().sum().sum()}")
print(f"Unique Values of the Target Variable 'state': {df['country'].unique()}")

Shape: (129971, 14)
Number of Missing Values: 204752
Unique Values of the Target Variable 'state': <StringArray>
[                 'Italy',               'Portugal',                     'US',
                  'Spain',                 'France',                'Germany',
              'Argentina',                  'Chile',              'Australia',
                'Austria',           'South Africa',            'New Zealand',
                 'Israel',                'Hungary',                 'Greece',
                'Romania',                 'Mexico',                 'Canada',
                      nan,                 'Turkey',         'Czech Republic',
               'Slovenia',             'Luxembourg',                'Croatia',
                'Georgia',                'Uruguay',                'England',
                'Lebanon',                 'Serbia',                 'Brazil',
                'Moldova',                'Morocco',                   'Peru',
                  

### Country

In [9]:
df1 = df[['description', 'country']]
df1 = df1[df1['description'].notnull() & df1['country'].notnull()]
df1.head()

,description,country
0,"Aromas include tropical fruit, broom, brimston...",Italy
1,"This is ripe and fruity, a wine that is smooth...",Portugal
2,"Tart and snappy, the flavors of lime flesh and...",US
3,"Pineapple rind, lemon pith and orange blossom ...",US
4,"Much like the regular bottling from 2012, this...",US


In [10]:
df1['country'].nunique()

43

In [27]:
df1 = df1[df1.groupby('country')['country'].transform('count') >= 5]
n_classes = df1['country'].nunique()

# Text Preprocessing

In [12]:
!python -m spacy download en_core_web_sm

  Using cached en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [13]:
nlp = spacy.load("en_core_web_sm")

In [14]:
def clean_text_lemmatize(text):
    # Lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Process with spaCy
    doc = nlp(text)

    # Lemmatize
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_.strip() != ""  # Remove empty tokens
        and len(token.lemma_) > 2      # Remove very short tokens (likely noise)
    ]

    return " ".join(tokens)

In [ ]:
# Apply text cleaning
df1["clean_description"] = df1["description"].apply(clean_text_lemmatize)

In [ ]:
label_encoder = LabelEncoder()
df1['country_encoded'] = label_encoder.fit_transform(df1['country'])

# Train/Validation/Test Split (70/15/15)

In [33]:
# Prepare features and labels
X = df1["clean_description"]
y = df1["country_encoded"]

In [34]:
# First split: Train (70%) vs Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,        # 30% for temp (will be split into val + test)
    stratify=y,           # Maintain class distribution
    random_state=42       # Reproducibility
)

# Second split: Val (15%) vs Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,        # 50% of temp = 15% of total
    stratify=y_temp,      # Maintain class distribution
    random_state=42
)

In [35]:
# Create DataFrames for each split
train_df = pd.DataFrame({
    'clean_description': X_train,
    'state': y_train
})

val_df = pd.DataFrame({
    'clean_description': X_val,
    'state': y_val
})

test_df = pd.DataFrame({
    'clean_description': X_test,
    'state': y_test
})

In [36]:
# Save to CSV
train_df.to_csv('train1.csv', index=False)
val_df.to_csv('val1.csv', index=False)
test_df.to_csv('test1.csv', index=False)

In [37]:
# Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3)
)

In [38]:
# Fit on training data and transform all sets
X_train_tfidf = vectorizer.fit_transform(X_train).toarray()
X_val_tfidf = vectorizer.transform(X_val).toarray()
X_test_tfidf = vectorizer.transform(X_test).toarray()

MemoryError: Unable to allocate 3.39 GiB for an array with shape (90930, 5000) and data type float64

# Building the Model

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], num_classes=n_classes, dropout=0.4):
        super(MLPClassifier, self).__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
# More robust ReviewDataset class
class ReviewDataset(Dataset):
    def __init__(self, features, labels):
        # Handle features conversion
        if hasattr(features, 'toarray'):  # Sparse matrix
            features = features.toarray()
        elif hasattr(features, 'values'):  # pandas Series/DataFrame
            features = features.values
        self.features = torch.FloatTensor(features)

        # Handle labels conversion
        if hasattr(labels, 'values'):  # pandas Series
            labels = labels.values
        elif hasattr(labels, 'flatten'):  # numpy array
            labels = labels.flatten()
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
# Create datasets
train_dataset = ReviewDataset(X_train_tfidf, y_train)
val_dataset = ReviewDataset(X_val_tfidf, y_val)
test_dataset = ReviewDataset(X_test_tfidf, y_test)

print(f"  Train dataset: {len(train_dataset):,} samples")
print(f"  Val dataset:   {len(val_dataset):,} samples")
print(f"  Test dataset:  {len(test_dataset):,} samples")

ValueError: too many dimensions 'str'

In [ ]:
# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Batch size: {batch_size}")
print(f"Batches - Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

In [ ]:
def objective_mlp_tfidf(trial):

    # MLP architecture parameters
    hidden_1 = trial.suggest_int('hidden_1', 128, 512)
    hidden_2 = trial.suggest_int('hidden_2', 64, 256)
    hidden_3 = trial.suggest_int('hidden_3', 32, 128)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)

    # Training parameters
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    l2 = trial.suggest_float("l2", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])

    epochs = trial.suggest_int('epochs', 5, 20)

    # Create TF-IDF features
    vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 3),
    )

    X_train_vec = vectorizer.fit_transform(X_train).toarray()
    X_val_vec = vectorizer.transform(X_val).toarray()

    # Create datasets and dataloaders
    train_dataset = ReviewDataset(X_train_vec, y_train)
    val_dataset = ReviewDataset(X_val_vec, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Create model
    model = MLPClassifier(
        input_dim=5000,
        hidden_dims=[hidden_1, hidden_2, hidden_3],
        dropout=dropout
    ).to(device)

    # Loss and optimizer
    class_counts = np.bincount(y_train)
    class_weights = torch.FloatTensor([1.0 / c for c in class_counts]).to(device)
    class_weights = class_weights * len(class_weights) / class_weights.sum()

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    # Train for limited epochs (for speed)
    num_epochs = 10
    best_val_f1 = 0
    patience = 3
    patience_counter = 0

    for epoch in range(num_epochs):
        # Training
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                _, preds = outputs.max(1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        val_f1 = f1_score(all_labels, all_preds)

        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    return best_val_f1

In [ ]:
# Run optimization
study_mlp_tfidf = optuna.create_study(
    direction='maximize',
    study_name='mlp_tfidf_optuna',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_mlp_tfidf.optimize(
    objective_mlp_tfidf,
    n_trials=10,
    show_progress_bar=True,
    n_jobs=1  # Don't parallelize to avoid GPU conflicts
)

In [ ]:
# Print results
print(f"Best Trial: #{study_mlp_tfidf.best_trial.number}")
print(f"Best Validation F1: {study_mlp_tfidf.best_trial.value:.4f}")
print(f"Best Hyperparameters:")
for key, value in study_mlp_tfidf.best_params.items():
    print(f"  {key:15s}: {value}")

In [ ]:

# Train final model with best parameters (full training)
# Create TF-IDF features with best params
vectorizer_mlp = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_vec = vectorizer_mlp.fit_transform(X_train).toarray()
X_val_vec = vectorizer_mlp.transform(X_val).toarray()
X_test_vec = vectorizer_mlp.transform(X_test).toarray()

# Create datasets
train_dataset = ReviewDataset(X_train_vec, y_train)
val_dataset = ReviewDataset(X_val_vec, y_val)
test_dataset = ReviewDataset(X_test_vec, y_test)

batch_size = study_mlp_tfidf.best_params['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Create model with best architecture
mlp_tfidf_optuna = MLPClassifier(
    input_dim=5000,
    hidden_dims=[
        study_mlp_tfidf.best_params['hidden_1'],
        study_mlp_tfidf.best_params['hidden_2'],
        study_mlp_tfidf.best_params['hidden_3']
    ],
    dropout=study_mlp_tfidf.best_params['dropout']
).to(device)

In [ ]:
# Loss and optimizer
class_counts = np.bincount(y_train)
class_weights = torch.FloatTensor([1.0 / c for c in class_counts]).to(device)
class_weights = class_weights * len(class_weights) / class_weights.sum()

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(
    mlp_tfidf_optuna.parameters(),
    lr=study_mlp_tfidf.best_params['lr'],
    weight_decay=1e-5
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

In [ ]:
# Training loop
num_epochs = 30
best_val_f1 = 0
patience = 7
patience_counter = 0

history = {'train_loss': [], 'val_f1': []}

for epoch in range(num_epochs):
    # Training
    mlp_tfidf_optuna.train()
    train_loss = 0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = mlp_tfidf_optuna(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    mlp_tfidf_optuna.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = mlp_tfidf_optuna(features)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    val_f1 = f1_score(all_labels, all_preds)

    history['train_loss'].append(train_loss)
    history['val_f1'].append(val_f1)

    scheduler.step(val_f1)

    print(f"Epoch {epoch+1:2d}/{num_epochs} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(mlp_tfidf_optuna.state_dict(), 'mlp_tfidf_optuna_best.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

In [ ]:
# Load best model
mlp_tfidf_optuna.load_state_dict(torch.load('mlp_tfidf_optuna_best.pth'))
print(f"Best validation F1: {best_val_f1:.4f}")

In [ ]:
# Evaluate on test set
def evaluate_mlp(model, loader):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)

            all_preds.extend(preds.cpu().tolist())
            all_probs.extend(probs[:, 1].cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)

    return {
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1': float(f1),
        'roc_auc': float(auc)
    }

mlp_tfidf_optuna_train = evaluate_mlp(mlp_tfidf_optuna, train_loader)
mlp_tfidf_optuna_val = evaluate_mlp(mlp_tfidf_optuna, val_loader)
mlp_tfidf_optuna_test = evaluate_mlp(mlp_tfidf_optuna, test_loader)

In [ ]:
print(f"Train Set:")
print(f"  Accuracy: {mlp_tfidf_optuna_train['accuracy']:.4f}")
print(f"  F1-Score: {mlp_tfidf_optuna_train['f1']:.4f}")

In [ ]:
print(f"Validation Set:")
print(f"  Accuracy: {mlp_tfidf_optuna_val['accuracy']:.4f}")
print(f"  F1-Score: {mlp_tfidf_optuna_val['f1']:.4f}")

In [ ]:
print(f"Test Set:")
print(f"  Accuracy: {mlp_tfidf_optuna_test['accuracy']:.4f}")
print(f"  F1-Score: {mlp_tfidf_optuna_test['f1']:.4f}")
print(f"  ROC AUC:  {mlp_tfidf_optuna_test['roc_auc']:.4f}")

In [ ]:
# Classification Report
mlp_tfidf_optuna.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        outputs = mlp_tfidf_optuna(features)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Failed (0)', 'Successful (1)']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Failed (0)', 'Successful (1)'],
            yticklabels=['Failed (0)', 'Successful (1)'])
plt.title('Confusion Matrix - MLP with TF-IDF (Optuna Optimized)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# AUC-ROC Curve
mlp_tfidf_optuna.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        outputs = mlp_tfidf_optuna(features)
        probs = torch.softmax(outputs, dim=1)
        all_probs.extend(probs[:, 1].cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = roc_auc_score(all_labels, all_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve\nMLP with TF-IDF (Optuna Optimized)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()